### Load images and dependencies

In [ ]:
from skimage import io
import csv
import pickle

classified_images = io.imread("Lysosome_masked_crops_1024.tif")
n_cl_images = classified_images.shape[0]

labels = []
with open("PhTau_histogram_clusters.csv") as csvfile:
    reader = csv.reader(csvfile)
    header = next(reader) # Skip header
    
    for row in reader:
        val = row[-1]
        if val == '0':
            labels.append(['0', 'dox+, PhTau+'])
        elif val == '1':
            labels.append(['1', 'dox+, PhTau-'])

# Handle untreated cells (label '2')
while len(labels) < n_cl_images:
    labels.append(['2', 'dox-'])

categories = {tuple(item) for item in labels}
categories = sorted(categories, key=lambda x: int(x[0]))
categories = [pair[1] for pair in categories]

scales = ["finest", "second", "combined"]

# Load coord_map
with open("processed_data/spot_coordinates.pkl", "rb") as f:
    coord_map = pickle.load(f)

# Load delaunay_objects_dict
with open("processed_data/delaunay_results.pkl", "rb") as f:
    delaunay_objects_dict = pickle.load(f)

new_images = io.imread("cell_gallery_1024x1024_6ch.tif")

### Classify based on PhTau intensity

In [ ]:
import numpy as np
import pandas as pd
from scipy import ndimage as ndi

analysis_results = []

for idx in range(classified_images.shape[0]):
    single_cell = classified_images[idx]
    phtau_ch = single_cell[:, :, 1]
    
    binary_mask = (single_cell > 0).any(axis=2)
    binary_mask = ndi.binary_closing(binary_mask) # Fill holes
    
    mean_intensity = np.mean(phtau_ch[binary_mask])
        
    analysis_results.append({
        'Mean_Intensity': mean_intensity,
        'True_Status': labels[idx][1] # Ground truth label from your metadata
    })

df_analysis = pd.DataFrame(analysis_results)

In [ ]:
# Print descriptive statistics for both groups
print(df_analysis.groupby('True_Status')['Mean_Intensity'].describe())

In [ ]:
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

# Compute False Positive Rate, True Positive Rate, and the corresponding thresholds
fpr, tpr, thresholds = roc_curve(df_analysis['True_Status'], df_analysis['Mean_Intensity'], pos_label='dox+, PhTau+')

# Calculate Youden's J statistic for each threshold
j_scores = tpr - fpr

# Locate the index of the highest J score
best_idx = np.argmax(j_scores)
optimal_threshold = thresholds[best_idx]

print(f"The mathematically optimal PhTau threshold is: {optimal_threshold:.2f}")

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# 1. Create overlapping histogram distribution plots
fig = px.histogram(
    df_analysis, 
    x="Mean_Intensity", 
    color="True_Status",
    barmode="overlay",
    color_discrete_map={0: "#3498db", 1: "#e74c3c"},
    labels={"Mean_Intensity": "PhTau Mean Intensity", "True_Status": "Ground Truth"},
    title="Interactive PhTau Distribution & Cutoff Threshold Analysis",
    opacity=0.6,
    nbins=50
)

# 2. Add a dynamic vertical line marking your calculated threshold
fig.add_vline(
    x=optimal_threshold, 
    line_width=3, 
    line_dash="dash", 
    line_color="black",
    annotation_text=f"Calculated Cutoff: {optimal_threshold:.2f}",
    annotation_position="top left"
)

# 3. Clean up layout styling for better scannability
fig.update_layout(
    xaxis_title="Mean Intensity Value",
    yaxis_title="Cell Count",
    legend_title="Cell Class",
    template="plotly_white",
    hovermode="x unified"
)

# 4. Display the interactive rendering
fig.show(renderer='notebook')

In [ ]:
THRESHOLD = optimal_threshold   # PhTau cutoff value

results = []

for image in range(new_images.shape[0]):
    # Extract the single cell crop array
    single_cell = new_images[image]
    
    # Isolate PhTau (Channel 2) and Mask (Channel 5) using 0-indexed python notation
    phtau_ch = single_cell[1, :, :]
    mask_ch  = single_cell[4, :, :]
        
    # Create a binary mask (True where mask channel is greater than 0)
    binary_mask = mask_ch > 0
    
    # Calculate mean intensity strictly within the masked cellular area
    if np.any(binary_mask):  # Prevent division by zero if mask is empty
        mean_intensity = np.mean(phtau_ch[binary_mask])
    else:
        mean_intensity = 0.0
        
    # Classify cell based on your hard threshold
    phtau_status = 0 if mean_intensity >= THRESHOLD else 1
    
    # Record data metrics
    results.append({
        'Cell_ID': image,
        'Mean_Intensity': mean_intensity,
        'PhTau_Status': phtau_status
    })

# Convert results into a clean dataframe
df = pd.DataFrame(results)

In [ ]:
# Inspect counts of positive vs negative cells
print(df['PhTau_Status'].value_counts())

# Save dataset
df.to_csv('gallery_phtau_ground_truth.csv', index=False)

In [ ]:
print(df.groupby('PhTau_Status')['Mean_Intensity'].describe())